# 1. Handle duplicates in fame_fixed

### [read] Check for duplicates in the resulting tables
Check for any duplicate registered_number values in fame_fixed and fame_derived  
Print a json in dirs.root_dir / "build" / "tmp" which is an array of objects, which contains properties:
- registered_number
- company_name
- industry_codes
- file_codes
- rows: the number of instances of this registered_number in this table
- all_other_properties_identical: a boolean indicating whether all other properties are identical across instances
- differing_properties: if and only iff all_other_properties_identical is False,
this will be a dict of the differing properties and their values across instances
In general within the for loop, use if not and then continue statements, rather than increasingly nested indentation

In [ ]:
import ibis
import json
import pandas as pd
from prompt_toolkit import keys


from utils.f_0_dirs import get_data_dirs
from typing import Any


tables = ["fame_fixed"] # , "fame_derived"] # "fame_yearly"]

dirs = get_data_dirs(segment="build")
db_path = dirs.output_dir / "fame_data.duckdb"
con = ibis.duckdb.connect(str(db_path))


time_start = pd.Timestamp.now()
def time_elapsed() -> str:
    elapsed = (pd.Timestamp.now() - time_start).total_seconds()
    return f"[{elapsed:5.1f}s] "


print(time_elapsed() + "⏱️ Starting duplicate detection and processing")


for table_name in tables:
    
    output_duplicate_paths = dirs.output_dir / f"duplicates_{table_name}.json"
    table = con.table(table_name)
    
    # 1. Identify columns to check
    ignore_cols = ["registered_number", "year"] if table_name == "fame_yearly" else ["registered_number"]
    check_cols = [c for c in table.columns if c not in ignore_cols]


    # 2. Build the Native Ibis Aggregation Dictionary
    aggs = {"rows": table.count()}
    for c in check_cols:
        # Cast to string, safely handle NULLs, and bundle into a DuckDB List
        aggs[f"{c}_vals"] = ibis.coalesce(table[c].cast("string"), "<NULL>").collect()


    print(f"{time_elapsed()}🚀 Executing native Ibis aggregation for {table_name}")


    # 3. Construct the Ibis Query Tree and Execute
    dup_counts = (
        table.group_by(ignore_cols)
        .aggregate(**aggs)
        .filter(ibis._.rows > 1)
    )
    count_dup_counts = dup_counts.count().execute()
    if count_dup_counts == 0:
        print(f"{time_elapsed()}✅ No duplicate ({', '.join(ignore_cols)}) values found in {table_name}.")
        continue
    print(f"{time_elapsed()}⚠️ Found {count_dup_counts:,} duplicate entries. Loading and processing")


    # Go back to the original table, throw away the 31.9 million healthy rows,
    # leaving ONLY the raw rows that belong to duplicates.
    filtered_dups = table.inner_join(dup_counts, ignore_cols)
    heavy_aggs = {}
    for c in check_cols:
        heavy_aggs[f"{c}_vals"] = ibis.coalesce(filtered_dups[c].cast("string"), "<NULL>").collect()
    heavy_agg = filtered_dups.group_by(ignore_cols + ["rows"]).aggregate(**heavy_aggs)
    grouped_df = heavy_agg.to_pyarrow().to_pandas()


    # 4. Fast Extraction: Iterate over the condensed array rows
    identical_list: list[dict[str, Any]] = []
    non_identical_list: list[dict[str, Any]] = []


    for _, row in grouped_df.iterrows():
        differing_properties = {}
        
        for col in check_cols:
            # We collected ALL values. We deduplicate them here in Python using set()
            # Because the arrays are tiny (usually 2-3 items), set() is functionally instant.
            unique_vals = list(set(row[f"{col}_vals"]))
            
            if len(unique_vals) > 1:
                # Convert back to standard None/str for clean JSON output
                clean_vals = [None if x == '<NULL>' else x for x in unique_vals]
                differing_properties[col] = clean_vals
                
        all_other_identical = len(differing_properties) == 0
        
        def get_summary_val(col_name) -> str | list | None:
            vals = row.get(col_name)
            if vals is None or len(vals) == 0:
                return None
            if len(vals) == 1:
                # If identical, just return the single string (or None)
                return None if pd.isna(vals[0]) else str(vals[0])
            else:
                # If they differ, return the array of strings so you can see both in the JSON
                return [None if pd.isna(x) else str(x) for x in vals]
        dup_record: dict[str, Any] = {
            "registered_number": str(row["registered_number"])
        }
        if table_name == "fame_fixed":
            dup_record.update({
                "company_name": get_summary_val("company_name"),
                "industry_codes": get_summary_val("industry_codes"),
                "file_codes": get_summary_val("file_codes")
            })
        elif table_name == "fame_yearly":
            dup_record.update({
                "year": int(row["year"])
            })


        dup_record.update({
            "rows": int(row["rows"]),
            "all_other_properties_identical": all_other_identical,
            "differing_properties": differing_properties
        })
        
        if all_other_identical:
            identical_list.append(dup_record)
        else:
            non_identical_list.append(dup_record)


    # 5. Summary and Output
    print(f"{time_elapsed()}❌ Found {len(non_identical_list):,} duplicate entries in {table_name} with differing properties.")


    output_duplicates = non_identical_list + identical_list


    with open(output_duplicate_paths, "w") as f:
        json.dump(output_duplicates, f, indent=4, default=str)
        print(f"{time_elapsed()}✅ Listed {len(output_duplicates):,} duplicates, saved to: {output_duplicate_paths}")

[  0.0s] ⏱️ Starting duplicate detection and processing
[  0.0s] 🚀 Executing native Ibis aggregation for fame_fixed
[  0.7s] ⚠️ Found 895,999 duplicate entries. Loading and processing
[150.7s] ❌ Found 123,841 duplicate entries in fame_fixed with differing properties.
[163.5s] ✅ Listed 895,999 duplicates, saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\duplicates_fame_fixed.json


### [process output] list categories of duplicates
- ex: 0 differences, only differ in 1 property

In [1]:
import json
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="build")


table_name = "fame_fixed"
output_dict = {}


print(f"Analyzing duplicates for table: {table_name}")


output_duplicate_paths = dirs.output_dir / f"duplicates_{table_name}.json"
duplicates_data = []
with open(output_duplicate_paths, "r") as f:
    duplicates_data = json.load(f)
duplicates_entries_ids = [entry["registered_number"] for entry in duplicates_data]
duplicates_ids_set = set(duplicates_entries_ids)
if len(duplicates_entries_ids) != len(duplicates_ids_set):
    raise ValueError(f"Duplicate registered_number values found in {output_duplicate_paths}. Please check the data.")
else:
    print(f"{len(duplicates_ids_set):,} unique registered_number values found in {output_duplicate_paths}.")


properties_set = set()
keys_dict = {}
for entry in duplicates_data:
    differing_props = entry.get("differing_properties", {})
    dp_frozen = frozenset(differing_props.keys())
    properties_set.add(dp_frozen)
    dp_key = ",".join(dp_frozen)
    if dp_key not in keys_dict:
        keys_dict[dp_key] = []
    keys_dict[dp_key].append(entry["registered_number"])

properties_list = sorted(list(properties_set), key=lambda x: (len(x), x))  # Sort by length first, then alphabetically
# True for i, x in enumerate(properties_list) if i == 0 or len(x) > len(properties_list[i - 1]) else False
start_of_increment = set({ x for i, x in enumerate(properties_list) if i == 0 or len(x) > len(properties_list[i - 1]) })
print(f"--- Unique properties ({len(properties_list)}): {properties_list}")
for frozen_p in properties_list:
    if frozen_p in start_of_increment:
        print(f"--- Dupe categories: {len(frozen_p)}")
    reg_nums = keys_dict.get(",".join(frozen_p), [])
    print(f"--- --- ({len(reg_nums):,}) {','.join(frozen_p)}: {', '.join(reg_nums)}")
    output_dict[",".join(frozen_p)] = reg_nums


Analyzing duplicates for table: fame_fixed
895,999 unique registered_number values found in C:\Users\lazyst\Files\ucl\Dissertation\build\output\duplicates_fame_fixed.json.
--- Unique properties (980): [frozenset(), frozenset({'branch_name'}), frozenset({'guo'}), frozenset({'ro_latitude'}), frozenset({'primary_trading_address'}), frozenset({'company_name'}), frozenset({'no_of_available_years'}), frozenset({'entity_type'}), frozenset({'latest_accounts_date'}), frozenset({'guo_nb'}), frozenset({'primary_uk_sic_2007_code', 'primary_uk_sic_2007_description'}), frozenset({'primary_trading_address_longitude', 'primary_trading_address_latitude'}), frozenset({'guo_nb', 'latest_accounts_date'}), frozenset({'entity_type', 'primary_trading_address'}), frozenset({'branch_name', 'company_name'}), frozenset({'guo', 'latest_accounts_date'}), frozenset({'ro_longitude', 'ro_latitude'}), frozenset({'ro_address', 'ro_full_postcode'}), frozenset({'branch_name', 'primary_trading_address'}), frozenset({'no_o

### [write] Start clearing files that are errored from main_dedupe
- 0 ("") differences: clear only ones with zero differing properties to start

In [ ]:
import ibis
table_name = "fame_fixed"

con = ibis.duckdb.connect(str(dirs.db_path))
table = con.table(table_name)
count_start = table.count().execute()
print(f"Starting with total {count_start:,} rows in {table_name}")

duplicate_ids = set({ entry["registered_number"] for entry in duplicates_data })
target_memtable = ibis.memtable({"registered_number": list(duplicate_ids)})
table_untouched = table.anti_join(target_memtable, "registered_number")
table_duplicates = table.inner_join(target_memtable, "registered_number")

for frozen_p in properties_list:
    frozen_len = len(frozen_p)
    if frozen_len >= 2:
        break

    output_key = ",".join(frozen_p)
    reg_nums_to_clear: list[str] = output_dict.get(output_key, [])
   
    if len(reg_nums_to_clear) <= 5:
        continue
       
    print(f"({frozen_len}) {''.join(frozen_p)}: Fetched {len(reg_nums_to_clear):,} unique duplicate IDs to resolve")
   
    if not reg_nums_to_clear:
        raise ValueError(f"No rows to clear from {table_name} with no differing properties.")

    target_ids = ibis.memtable({"registered_number": reg_nums_to_clear})
    unaffected_rows = table_duplicates.anti_join(target_ids, "registered_number")
    target_rows = table_duplicates.inner_join(target_ids, "registered_number")

    if frozen_len == 0:
        resolved_duplicates = target_rows.distinct()
    else:
        agg_exprs = {f"{col}_count": target_rows[col].count() for col in frozen_p}
        counts = target_rows.group_by("registered_number").aggregate(**agg_exprs)
       
        joined_target = target_rows.inner_join(counts, "registered_number")
       
        can_collapse = None
        for col in frozen_p:
            condition = joined_target[f"{col}_count"] <= 1
            can_collapse = condition if can_collapse is None else (can_collapse & condition)
           
        count_cols = [f"{col}_count" for col in frozen_p]
       
        untouched = joined_target.filter(~can_collapse).drop(*count_cols)
        to_collapse = joined_target.filter(can_collapse).drop(*count_cols)
       
        other_cols = [c for c in table.columns if c not in frozen_p]
        collapse_aggs = {col: to_collapse[col].max() for col in frozen_p}
        collapsed = to_collapse.group_by(other_cols).aggregate(**collapse_aggs)
       
        resolved_duplicates = untouched.select(table.columns).union(collapsed.select(table.columns))

    table_duplicates = unaffected_rows.union(resolved_duplicates)

final_table = table_untouched.union(table_duplicates)
count_end = final_table.count().execute()
rows_removed = count_start - count_end

print(f"--- Net duplicate rows removed: {rows_removed:,}")
if rows_removed == 0:
    raise ValueError(f"Unexpected negative row count difference for {table_name}.")
print(f"✅ {table_name} had {count_start:,} rows before clearing, now has {count_end:,} rows.")

Starting with total 9,283,479 rows in fame_fixed
(0) : Fetched 772,158 unique duplicate IDs to resolve
(1) branch_name: Fetched 31 unique duplicate IDs to resolve
(1) guo: Fetched 2,164 unique duplicate IDs to resolve
(1) primary_trading_address: Fetched 101,037 unique duplicate IDs to resolve
(1) company_name: Fetched 13 unique duplicate IDs to resolve
(1) entity_type: Fetched 247 unique duplicate IDs to resolve
(1) latest_accounts_date: Fetched 574 unique duplicate IDs to resolve
(1) guo_nb: Fetched 3,632 unique duplicate IDs to resolve
--- Net duplicate rows removed: 1,067,953


In [4]:
con.create_table(f"{table_name}_clean", final_table, overwrite=True)
con.drop_table(table_name)
con.create_table(table_name, con.table(f"{table_name}_clean"), overwrite=True)
con.drop_table(f"{table_name}_clean")
print(f"✅ {table_name} confirming has {con.table(table_name).count().execute():,} rows")

✅ fame_fixed confirming has 8,215,526 rows


# 2. Handle duplicates in fame_yearly

### [read] Check for conflicts in fame_yearly

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="build")

sample_size = 2000
table_name = "fame_yearly_filtered"

db_path = dirs.output_dir / "fame_data.duckdb"
con = ibis.duckdb.connect(str(db_path))
t = con.table(table_name)
keys = ["registered_number", "year"]
print(f"🔍 Sampling {sample_size:,} keys from {table_name} & materializing to Pandas...")


# Chain ops: Sample random keys -> Join full table -> Count rows -> Pull to Pandas
dups = (
    t.inner_join(t.select(keys).distinct().order_by(ibis.random()).limit(sample_size), keys)
    .group_by(keys).aggregate(row_count=ibis._.count())
    .to_pandas()
)


print(f"✅ Sampled {len(dups):,} rows from {table_name}.")


if not dups.empty:
    print("📈 Rows per key distribution:\n", dups['row_count'].value_counts().sort_index().to_string())
    print("\n🔎 Top 5 Duplicate Keys:\n", dups.head(5).to_string(index=False))
    
    # Raw dump of the first duplicate found
    eg = dups.iloc[0]
    print(f"\n🔬 Raw Dump for {eg.registered_number} ({eg.year}):")
    dump = t.filter((t.registered_number == eg.registered_number) & (t.year == eg.year)).to_pandas()
    print(dump.dropna(axis=1, how='all').to_string(index=False))
else:
    print("✅ Zero duplicates found in the sample.")

🔍 Sampling 2,000 keys from fame_yearly_filtered & materializing to Pandas...
✅ Sampled 2,000 rows from fame_yearly_filtered.
📈 Rows per key distribution:
 row_count
1    1935
2      61
3       4

🔎 Top 5 Duplicate Keys:
 registered_number  year  row_count
         02108152  2016          1
         02698057  2016          1
         00584915  2016          1
         03969379  2016          1
         08009760  2016          1

🔬 Raw Dump for 02108152 (2016):
registered_number  year  consolidated  turnover  shareholders_funds  profit_loss_pretax  employees  tangibles  tangibles_land_and_buildings  tangibles_land_freehold  tangibles_land_leasehold  tangibles_fixt_fit  tangibles_plant_and_vehicles  tangibles_plant  tangibles_vehicles  investments_other  fixed_total  liabilities  total_assets  liabilites_lt      cos  admin_expenses  interest_paid  profit_loss_pretax2    tax  depreciation  remuneration_employees   wages  social_security_costs  pensions_costs  renumeration_directors   ebitd

### [write] max row algorithm to crush staggered fame_yearly rows

In [2]:
# --- Configuration ---
t = con.table(table_name)
keys = ["registered_number", "year"]
metric_cols = [c for c in t.columns if c not in keys]

con.raw_sql("PRAGMA memory_limit='12GB'")

start_count = t.count().execute()
print(f"🚀 Starting consolidation for '{table_name}' with {start_count:,} rows")

# --- 1. Build the Lazy Aggregation AST ---
merge_aggs = {col: t[col].max() for col in metric_cols}
merged_table_expr = t.group_by(keys).aggregate(**merge_aggs)

# --- 2. Create Empty Target Table ---
print("🏗️ Creating empty consolidated table...")
empty_schema = merged_table_expr.filter(t.year == -9999)
con.create_table(f"{table_name}_consolidated", empty_schema, overwrite=True)

# --- 3. Process in Memory-Safe Batches ---
unique_years = t.select('year').distinct().execute()['year'].tolist()
earliest_year = min(unique_years)
latest_year = max(unique_years)
print(f"🔄 Starting batched aggregation across {len(unique_years)} years from {earliest_year} to {latest_year}:")

for y in sorted(unique_years):
    print(f"   Processing Year: {y}...")
    
    # Filter the lazy expression to just THIS year
    year_chunk = merged_table_expr.filter(t.year == y)
    
    # Execute the chunk and APPEND it directly to disk
    con.insert(f"{table_name}_consolidated", year_chunk)

row_count = con.table(f"{table_name}_consolidated").count().execute()
print(f"✅ Created consolidated temporary table. {row_count:,} rows.")

🚀 Starting consolidation for 'fame_yearly_filtered' with 1,171,378 rows
🏗️ Creating empty consolidated table...
🔄 Starting batched aggregation across 19 years from 2006 to 2024:
   Processing Year: 2006...
   Processing Year: 2007...
   Processing Year: 2008...
   Processing Year: 2009...
   Processing Year: 2010...
   Processing Year: 2011...
   Processing Year: 2012...
   Processing Year: 2013...
   Processing Year: 2014...
   Processing Year: 2015...
   Processing Year: 2016...
   Processing Year: 2017...
   Processing Year: 2018...
   Processing Year: 2019...
   Processing Year: 2020...
   Processing Year: 2021...
   Processing Year: 2022...
   Processing Year: 2023...
   Processing Year: 2024...
✅ Created consolidated temporary table. 1,128,490 rows.


In [3]:
# How many rows were removed?
removed_count = start_count - row_count
print(f"ℹ️ Removed {removed_count:,} duplicate rows from {table_name}")


# --- 4. Safe Overwrite ---
print("💾 Overwriting table with consolidated data...")
con.drop_table(table_name)
con.create_table(table_name, con.table(f"{table_name}_consolidated"), overwrite=True)
con.drop_table(f"{table_name}_consolidated")


print(f"✅ Consolidation complete. Final row count: {con.table(table_name).count().execute():,}")

ℹ️ Removed 42,888 duplicate rows from fame_yearly_filtered
💾 Overwriting table with consolidated data...
✅ Consolidation complete. Final row count: 1,128,490
